In [1]:
# Polariton Disorder — Computation Notebook

# Computes the disorder self-energy on a uniform momentum mesh and
# reconstructs the on-shell Q(k, eta) by sweeping an external energy E_ext.
# Results are saved to `Results/` as `.npy` arrays with `_meta.json` sidecars.

# **Workflow**
# 1. Define a base `Params` and any sweep axes in *Cell 2 — Parameters & sweep*.
# 2. Run *Cell 3 — Sweep helpers* to expand the parameter combinations.
# 3. Run *Cell 4 — Kernel mesh* to build and save K(q,k) on the uniform mesh.
# 4. Run *Cell 5 — Sigma sweep* to compute Sigma(k, E_ext, eta) and reduce
#    it to on-shell Q(k, eta) via the root of Re[E_ext - bare(k) - Sigma].
# 5. Run *Cell 6 — Diagnostic* to plot E_k'(k) for each eta.
# 6. Open `Visualisations.ipynb` to plot saved Q results.


In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from polaritons.parameters import Params, DEFAULT_PARAMS

# ---------------------------------------------------------------------------
# Base parameter set  (modify here to change the physics)
# ---------------------------------------------------------------------------

base = Params(
	# Material (GaAs)
	E_bind       = 4.2e-3,          # eV  -- exciton binding energy
	E_gap_bare   = 1.519,             # eV  -- bare band gap
	m_e          = 3.692e-13,        # eV s^2/m^2
	m_h          = 9.372e-13,        # eV s^2/m^2
	m_rest       = 5.68e-12,        # eV s^2/m^2

	# Polariton / cavity
	Omega        = 1.4e-2,          # eV  -- Rabi splitting
	m_prime      = 1,
	n_refr       = 3.0,             # cavity refractive index
	N_qw         = 1,               # number of quantum wells

	# Disorder
	D_0          = 2.26e-20,        # eV^2 m^2
	xi           = 20e-9,           # m  -- correlation length

	# Thermodynamics
	T            = 20.0,            # K
	concentration = 0.3e12,         # m^-2
	g_ex         = 12e-18,          # eV m^2
)

# ---------------------------------------------------------------------------
# Parameter sweep axes
#
# Each entry is  (param_name, list_of_values).
# All combinations are explored.  Use a single-element list to hold a param fixed.
# ---------------------------------------------------------------------------

SWEEP = {
	"xi"    : [20e-9],   # correlation length (m)
	"D_0"   : [2.26e-20],              # disorder strength -- single value keeps it fixed
}

# eta grid -- disorder amplitude sweep (always included)
eta_grid = np.round(np.linspace(0.0, 2.0, 3), 9)

print(f"Base params:\n  E_bind={base.E_bind*1e3:.1f} meV,  xi={base.xi*1e9:.0f} nm,  "
	f"T={base.T:.0f} K,  D_0={base.D_0:.2e} eV^2m^2,  m_prime={base.m_prime:g}")
print(f"Sweep axes: {list(SWEEP.keys())}")
print(f"eta_grid: {len(eta_grid)} points from {eta_grid[0]} to {eta_grid[-1]}")


Base params:
  E_bind=4.2 meV,  xi=20 nm,  T=20 K,  D_0=2.26e-20 eV^2m^2,  m_prime=1
Sweep axes: ['xi', 'D_0']
eta_grid: 3 points from 0.0 to 2.0


In [3]:
import itertools
from dataclasses import replace

def build_sweep_params(base: Params, sweep: dict) -> list[Params]:
	"""
	Return a list of Params objects, one for each combination of sweep values.

	Parameters
	----------
	base   : base Params (SI units)
	sweep  : dict mapping field names to lists of values to sweep over

	Example
	-------
	sweep = {"xi": [10e-9, 20e-9], "T": [10.0, 20.0]}
	→ 4 Params objects covering all (xi, T) combinations
	"""
	keys   = list(sweep.keys())
	values = list(sweep.values())
	combos = list(itertools.product(*values))

	params_list = []
	for combo in combos:
		overrides = dict(zip(keys, combo))
		p = replace(base, **overrides)
		params_list.append(p)
	return params_list


sweep_params_si      = build_sweep_params(base, SWEEP)
sweep_params_natural = [p.to_natural() for p in sweep_params_si]

print(f"{len(sweep_params_si)} parameter set(s) in sweep:")
for i, p in enumerate(sweep_params_si):
	print(f"  [{i}]  xi={p.xi*1e9:.0f} nm  D_0={p.D_0:.2e}  T={p.T:.0f} K")


1 parameter set(s) in sweep:
  [0]  xi=20 nm  D_0=2.26e-20  T=20 K


In [4]:
from polaritons.kernel import make_kernel_gaussian, make_kernel_nongaussian, find_kernel_truncation
from polaritons.grid   import uniform_grid_and_weights
from polaritons.io     import save_result, make_sweep_stem
from polaritons.units  import k_cm_to_nat, k_nat_to_cm

# ---------------------------------------------------------------------------
# Grid configuration (per-kernel)
# ---------------------------------------------------------------------------

SWEEP_SCHEMA = "sigma_uniform_v1"

KERNEL_FACTORIES = {
	"nongaussian": make_kernel_nongaussian,
	"gaussian"   : make_kernel_gaussian,
}

# Truncation threshold: pick t per-kernel so that |K(t,t)| <= KERNEL_THRESHOLD * |K(0,0)|,
# lower-bounded by K_MIN_CM (cm^-1). Each kernel (non-Gaussian shared; Gaussian
# per ξ) uses its own t_local, and builds its own Picard grid on [0, t_local].
KERNEL_THRESHOLD = 0.1
K_MIN_CM         = 40_000.0      # lower bound on truncation in cm^-1
N_PICARD         = 3_000         # number of uniform grid points

kernel_jobs = []


def kernel_run_label(kernel_type, p_si, xi_independent=False):
	parts = [kernel_type]
	if not xi_independent:
		parts.append(f"ξ={p_si.xi*1e9:.0f} nm")
	parts.append(f"D_0={p_si.D_0:.2e}")
	parts.append(f"m_prime={p_si.m_prime:g}")
	return ", ".join(parts)


# ---------------------------------------------------------------------------
# Step 1: enumerate kernel jobs (non-Gaussian shared; Gaussian per ξ).
# ---------------------------------------------------------------------------
pending_jobs = []
pending_jobs.append(dict(
	p_si=base, p_nat=base.to_natural(), kernel_type="nongaussian",
	sweep_index=None, xi_independent=True,
))
for idx, (p_si, p_nat) in enumerate(zip(sweep_params_si, sweep_params_natural)):
	pending_jobs.append(dict(
		p_si=p_si, p_nat=p_nat, kernel_type="gaussian",
		sweep_index=idx, xi_independent=False,
	))

# ---------------------------------------------------------------------------
# Step 2: discover each kernel's per-kernel truncation t_local.
# ---------------------------------------------------------------------------
print("Discovering per-kernel truncations...")
for job in pending_jobs:
	p_nat       = job["p_nat"]
	K_fn        = KERNEL_FACTORIES[job["kernel_type"]](p_nat, n_gauss=96)
	k_lower_nat = float(k_cm_to_nat(K_MIN_CM, p_nat))
	t_nat       = find_kernel_truncation(K_fn, threshold=KERNEL_THRESHOLD, k_lower=k_lower_nat)
	t_cm        = float(k_nat_to_cm(t_nat, p_nat))
	job["K_fn"]        = K_fn
	job["t_nat_local"] = t_nat
	job["t_cm_local"]  = t_cm
	job["k_lower_nat"] = k_lower_nat
	print(f"  {kernel_run_label(job['kernel_type'], job['p_si'], job['xi_independent'])}: "
	    f"t_local = {t_nat:.4g} (nat) = {t_cm:.3g} cm^-1")

per_job_t_nat = {
	f"{job['kernel_type']}|sweep={job['sweep_index']}": float(job["t_nat_local"])
	for job in pending_jobs
}

# ---------------------------------------------------------------------------
# Step 3: build a per-kernel grid and compute each kernel mesh on its own grid.
# ---------------------------------------------------------------------------
for job in pending_jobs:
	p_si           = job["p_si"]
	p_nat          = job["p_nat"]
	kernel_type    = job["kernel_type"]
	sweep_index    = job["sweep_index"]
	xi_independent = job["xi_independent"]
	K_fn           = job["K_fn"]
	t_nat_local    = job["t_nat_local"]
	print(f"\n-- {kernel_run_label(kernel_type, p_si, xi_independent)} --")

	q_picard, weights_picard = uniform_grid_and_weights(t_nat_local, N_PICARD)
	print(f"  Grid: N={N_PICARD}, K_max={t_nat_local:.4g}, dq={q_picard[1]-q_picard[0]:.4g}")
	print(f"  Computing kernel mesh ({N_PICARD}x{N_PICARD}) on this kernel's grid...")
	K_mesh = K_fn(q_picard, q_picard)

	extra_meta = {
		"kernel_type"        : kernel_type,
		"xi_independent"     : bool(xi_independent),
		"N_picard"           : N_PICARD,
		"K_domain_max"       : float(t_nat_local),
		"K_domain_max_local" : float(t_nat_local),
		"K_domain_max_shared": False,
		"per_job_t_nat"      : per_job_t_nat,
		"kernel_threshold"   : KERNEL_THRESHOLD,
		"k_min_cm"           : K_MIN_CM,
		"k_min_nat"          : job["k_lower_nat"],
		"q_picard"           : q_picard.tolist(),
		"sweep_index"        : sweep_index,
		"xi_m"               : None if xi_independent else p_si.xi,
		"m_prime"            : p_si.m_prime,
		"calculation_units"  : "natural",
		"sweep_schema"       : SWEEP_SCHEMA,
	}
	stem = make_sweep_stem("K", p_nat, extra={
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"sweep_schema"   : SWEEP_SCHEMA,
	})
	save_result(K_mesh, "Results/integrand_meshes", stem, p_nat, extra_meta)
	kernel_jobs.append({
		"sweep_index"    : sweep_index,
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"stem"           : stem,
		"p_si"           : p_si,
		"p_nat"          : p_nat,
		"K_domain_max"   : float(t_nat_local),
		"q_picard"       : q_picard,
		"weights_picard" : weights_picard,
	})

print(f"\nAll kernels saved.  Jobs: {[(job['kernel_type'], job['stem']) for job in kernel_jobs]}")


Discovering per-kernel truncations...
  nongaussian, D_0=2.26e-20, m_prime=1: t_local = 6.451 (nat) = 4.62e+06 cm^-1
  gaussian, ξ=20 nm, D_0=2.26e-20, m_prime=1: t_local = 2.235 (nat) = 1.6e+06 cm^-1

-- nongaussian, D_0=2.26e-20, m_prime=1 --
  Grid: N=3000, K_max=6.451, dq=0.002151
  Computing kernel mesh (3000x3000) on this kernel's grid...
Saved  K_4c2d16.npy  [3000, 3000] float64

-- gaussian, ξ=20 nm, D_0=2.26e-20, m_prime=1 --
  Grid: N=3000, K_max=2.235, dq=0.0007454
  Computing kernel mesh (3000x3000) on this kernel's grid...
Saved  K_d69019.npy  [3000, 3000] float64

All kernels saved.  Jobs: [('nongaussian', 'K_4c2d16'), ('gaussian', 'K_d69019')]


In [5]:
from polaritons.sigma import sweep_sigma, find_E_k_prime, assemble_Q, refine_sigma_per_k
from polaritons.io    import load_result, save_result, make_sweep_stem

# ---------------------------------------------------------------------------
# Sigma sweep settings
# ---------------------------------------------------------------------------

PICARD_TOL      = 1e-6
PICARD_MAX_ITER = 10_000
PICARD_W        = 0.99
PICARD_VERBOSE  = True

# Pass 1: broad window centred on the exciton band bottom (which is 0 in the
# band-bottom-relative convention used by polaritons.dispersion). One Picard
# solve per (η, E_ext) gives Σ(k, E_ext) for all k at once.
E_EXT_HALF_WIDTH_OMEGA = 50.0   # window is [-N*Ω, +N*Ω]
N_E_EXT                = 321

# Pass 2: per-(η, k) adaptive refinement. For every k we build a tight
# uniform window of width 2 * PASS2_HALF_WIDTH_OMEGA * Ω centred on
# pass-1's E_k_prime estimate and run N_E_PASS2 Picard solves on that window
# (one solve per (k, j) pair, so cost scales as N * N_E_PASS2 per η).
# Where pass 1 failed (NaN), refine_sigma_per_k falls back to the bare
# exciton energy so no NaN is propagated.
RUN_SECOND_PASS         = False
PASS2_HALF_WIDTH_OMEGA  = 1.0   # per-k window half-width in units of Ω
N_E_PASS2               = 21    # samples per per-k window (>= 2)

SAVE_SIGMA = True

for job in kernel_jobs:
	kernel_type    = job["kernel_type"]
	k_stem         = job["stem"]
	p_si           = job["p_si"]
	p_nat          = job["p_nat"]
	sweep_index    = job["sweep_index"]
	xi_independent = job["xi_independent"]
	print(f"\n-- Sigma sweep, {kernel_run_label(kernel_type, p_si, xi_independent)} --")

	K_mesh, k_meta = load_result("Results/integrand_meshes", k_stem)
	q            = np.array(k_meta["q_picard"])
	N_pic        = len(q)
	K_domain_max = float(k_meta["K_domain_max"])
	weights      = uniform_grid_and_weights(K_domain_max, N_pic)[1]

	E_ext_grid = np.linspace(
		-E_EXT_HALF_WIDTH_OMEGA * p_nat.Omega,
		+E_EXT_HALF_WIDTH_OMEGA * p_nat.Omega,
		N_E_EXT,
	)
	print(f"  Pass 1 E_ext window: [{E_ext_grid[0]:.4g}, {E_ext_grid[-1]:.4g}] "
	      f"(natural energy units), N={N_E_EXT}")
	print(f"  eta grid    : {len(eta_grid)} points")

	# --- Pass 1 -------------------------------------------------------
	Sigma_pass1, iters_pass1 = sweep_sigma(
		p_nat, K_mesh, q, weights, E_ext_grid, eta_grid,
		tol=PICARD_TOL, max_iter=PICARD_MAX_ITER, w=PICARD_W,
		verbose=PICARD_VERBOSE,
	)
	print(f"  Pass 1 iters: mean={iters_pass1.mean():.1f}, max={iters_pass1.max()}, "
	      f"hit max_iter on {(iters_pass1 == PICARD_MAX_ITER).sum()} solves")

	E_k_prime_pass1 = find_E_k_prime(Sigma_pass1, q, E_ext_grid, p_nat)

	# --- Pass 2 (per-(η, k) adaptive refinement) -------------------
	if RUN_SECOND_PASS:
		half_width_nat = PASS2_HALF_WIDTH_OMEGA * p_nat.Omega

		E_k_prime_pass2     = np.full((len(eta_grid), N_pic), np.nan, dtype=float)
		Q_results_pass2     = np.full((len(eta_grid), N_pic), np.nan + 1j * np.nan, dtype=complex)
		Sigma_pass2_col     = np.zeros((len(eta_grid), N_E_PASS2, N_pic), dtype=complex)
		E_per_k_pass2       = np.zeros((len(eta_grid), N_E_PASS2, N_pic), dtype=float)
		iters_pass2         = np.zeros((len(eta_grid), N_E_PASS2, N_pic), dtype=int)

		for ei, eta in enumerate(eta_grid):
			Ek, Q_eta, Sig_col, E_pk, it_pk = refine_sigma_per_k(
				p_nat, K_mesh, q, weights,
				eta=float(eta),
				E_seed=E_k_prime_pass1[ei],
				half_width=half_width_nat,
				n_E_pk=N_E_PASS2,
				tol=PICARD_TOL, max_iter=PICARD_MAX_ITER, w=PICARD_W,
				verbose=PICARD_VERBOSE,
			)
			E_k_prime_pass2[ei] = Ek
			Q_results_pass2[ei] = Q_eta
			Sigma_pass2_col[ei] = Sig_col
			E_per_k_pass2[ei]   = E_pk
			iters_pass2[ei]     = it_pk
			nan_count           = int(np.isnan(Ek).sum())
			print(f"  Pass 2 η={eta:.2f}: window half-width = {half_width_nat:.4g} (nat), "
			      f"N_E_PASS2={N_E_PASS2}, iters mean={it_pk.mean():.1f}, "
			      f"NaN E_k' = {nan_count}")

		Sigma_arr = Sigma_pass2_col
		E_k_prime = E_k_prime_pass2
		Q_results = Q_results_pass2
	else:
		Sigma_arr       = Sigma_pass1
		E_k_prime       = E_k_prime_pass1
		Q_results       = assemble_Q(Sigma_arr, E_k_prime, E_ext_grid)
		E_per_k_pass2   = None
		iters_pass2     = None

	nan_per_eta = np.isnan(E_k_prime).sum(axis=1)
	if nan_per_eta.any():
		print(f"  WARN: NaN E_k' counts per eta: {nan_per_eta.tolist()} "
		      "(consider widening E_ext or pass-2 window)")

	extra_q = {
		"eta_grid"        : eta_grid.tolist(),
		"q_picard"        : q.tolist(),
		"E_ext_grid"      : E_ext_grid.tolist(),    # pass-1 grid
		"kernel_stem"     : k_stem,
		"kernel_type"     : kernel_type,
		"xi_independent"  : bool(xi_independent),
		"K_domain_max"    : K_domain_max,
		"kernel_threshold": float(k_meta.get("kernel_threshold", 0.0)) or None,
		"k_min_cm"        : float(k_meta.get("k_min_cm", 0.0)) or None,
		"picard_tol"      : PICARD_TOL,
		"picard_max_iter" : PICARD_MAX_ITER,
		"picard_w"        : PICARD_W,
		"sweep_index"     : sweep_index,
		"xi_m"            : None if xi_independent else p_si.xi,
		"m_prime"         : p_si.m_prime,
		"calculation_units": "natural",
		"sweep_schema"    : SWEEP_SCHEMA,
		"saved_sigma"     : bool(SAVE_SIGMA),
		"run_second_pass" : bool(RUN_SECOND_PASS),
		"pass2_scheme"    : "per_k_adaptive" if RUN_SECOND_PASS else None,
		"pass2_half_width_omega": PASS2_HALF_WIDTH_OMEGA if RUN_SECOND_PASS else None,
		"n_e_pass2"       : N_E_PASS2 if RUN_SECOND_PASS else None,
		"convention"      : "band_bottom_relative",
	}
	q_stem = make_sweep_stem("Q", p_nat, extra={
		"kernel_stem"    : k_stem,
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"sweep_schema"   : SWEEP_SCHEMA,
	})
	save_result(Q_results, "Results/Q_results", q_stem, p_nat, extra_q)

	if SAVE_SIGMA:
		import os
		base_dir = "Results/Q_results"
		np.save(os.path.join(base_dir, f"{q_stem}_Sigma.npy"),            Sigma_arr)
		np.save(os.path.join(base_dir, f"{q_stem}_E_k_prime.npy"),        E_k_prime)
		np.save(os.path.join(base_dir, f"{q_stem}_Sigma_pass1.npy"),      Sigma_pass1)
		np.save(os.path.join(base_dir, f"{q_stem}_E_k_prime_pass1.npy"),  E_k_prime_pass1)
		np.save(os.path.join(base_dir, f"{q_stem}_E_ext_grid_pass1.npy"), E_ext_grid)
		if RUN_SECOND_PASS:
			np.save(os.path.join(base_dir, f"{q_stem}_E_per_k_pass2.npy"),  E_per_k_pass2)
			np.save(os.path.join(base_dir, f"{q_stem}_iters_pass2.npy"),    iters_pass2)

	job["q"]         = q
	job["E_ext"]     = E_per_k_pass2 if RUN_SECOND_PASS else E_ext_grid
	job["Sigma"]     = Sigma_arr
	job["E_k_prime"] = E_k_prime
	job["Q"]         = Q_results

print("\nAll Sigma sweeps complete.")



-- Sigma sweep, nongaussian, D_0=2.26e-20, m_prime=1 --
  Pass 1 E_ext window: [-166.7, 166.7] (natural energy units), N=321
  eta grid    : 3 points
  iter     0  |ΔQ| = 2.816e+01
  iter     0  |ΔQ| = 1.751e-01
  iter     0  |ΔQ| = 1.773e-01
  iter     0  |ΔQ| = 1.795e-01
  iter     0  |ΔQ| = 1.818e-01
  iter     0  |ΔQ| = 1.841e-01
  iter     0  |ΔQ| = 1.865e-01
  iter     0  |ΔQ| = 1.889e-01
  iter     0  |ΔQ| = 1.914e-01
  iter     0  |ΔQ| = 1.939e-01
  iter     0  |ΔQ| = 1.964e-01
  iter     0  |ΔQ| = 1.990e-01
  iter     0  |ΔQ| = 2.017e-01
  iter     0  |ΔQ| = 2.044e-01
  iter     0  |ΔQ| = 2.072e-01
  iter     0  |ΔQ| = 2.100e-01
  iter     0  |ΔQ| = 2.129e-01
  iter     0  |ΔQ| = 2.158e-01
  iter     0  |ΔQ| = 2.188e-01
  iter     0  |ΔQ| = 2.219e-01
  iter     0  |ΔQ| = 2.250e-01
  iter     0  |ΔQ| = 2.282e-01
  iter     0  |ΔQ| = 2.315e-01
  iter     0  |ΔQ| = 2.349e-01
  iter     0  |ΔQ| = 2.383e-01
  iter     0  |ΔQ| = 2.418e-01
  iter     0  |ΔQ| = 2.453e-01
  iter     0